# 02 Noise Covariance

This notebook creates recording-specific noise covariance matrices.

Default mode:

```python
NOISE_COV_MODE = "erm"
```

ERM matching is configured under `empty_room.matching`. For the current project, `meas_date_nearest` is the recommended strategy because subject recordings and empty-room recordings both contain valid FIF `meas_date` metadata.

Fallback modes for technical testing:

- `epochs_baseline`
- `adhoc`

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    noise_covariance_input_overview_to_dataframe,
    noise_covariance_results_to_dataframe,
    source_forward_config_to_dataframe,
    write_noise_covariances_for_recordings,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## MNE logging

In [ ]:
mne.set_log_level("WARNING")

## Selection

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

RUN_COVARIANCE_QC_PLOTS = True
MAX_COVARIANCE_QC_PLOTS = None  # None = alle, oder z. B. 3 zum Testen

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)

## Overwrite policy

Default:

```python
OVERWRITE_STEPS = []
```

To recompute covariance files:

```python
OVERWRITE_STEPS = ["noise_covariance"]
```

In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "noise_covariance",
            "overwrite": should_overwrite("noise_covariance", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "noise_covariance",
                OVERWRITE_STEPS,
            ),
        }
    ]
)

## Covariance mode and method

Use ERM as the default source-modeling covariance input. `COV_METHOD = "empirical"` keeps the batch run numerically quiet and stable.

In [ ]:
NOISE_COV_MODE = config.source.noise_cov.mode
COV_METHOD = "empirical"
RANK = None

pd.concat(
    [
        source_forward_config_to_dataframe(config),
        pd.DataFrame(
            [
                {
            "configured_noise_cov_mode": config.source.noise_cov.mode,
            "active_noise_cov_mode": NOISE_COV_MODE,
            "method": COV_METHOD,
            "rank": RANK,
            "empty_room_strategy": config.empty_room.matching.strategy,
            "max_time_diff_hours": config.empty_room.matching.max_time_diff_hours,
            "fallback_strategy": config.empty_room.matching.fallback_strategy,
                }
            ]
        ),
    ],
    axis=1,
)

## Input overview

For `NOISE_COV_MODE = "erm"`, this table shows the matched empty-room recording and the matching metadata.

Important columns:

- `recording_meas_date`
- `selected_erm_session`
- `selected_erm_meas_date`
- `time_diff_hours`
- `match_strategy`
- `data_input`

In [ ]:
noise_cov_policy = existing_output_policy_for_step(
    "noise_covariance",
    OVERWRITE_STEPS,
)

noise_cov_overview = noise_covariance_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=noise_cov_policy,
    mode=NOISE_COV_MODE,
)

noise_cov_overview

## Matching summary

In [ ]:
columns = [
    "subject",
    "session",
    "task",
    "run",
    "status",
    "message",
    "mode",
    "data_input_kind",
    "match_strategy",
    "recording_meas_date",
    "selected_erm_session",
    "selected_erm_meas_date",
    "time_diff_hours",
    "data_input",
    "cov_path",
]

existing_columns = [column for column in columns if column in noise_cov_overview.columns]
noise_cov_overview[existing_columns]

## Status summary

In [ ]:
if noise_cov_overview.empty:
    pd.DataFrame()
else:
    (
        noise_cov_overview
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Ready jobs

In [ ]:
ready_noise_cov_jobs = noise_cov_overview.query("status == 'ready'").copy()

columns = [
    "subject",
    "session",
    "task",
    "run",
    "mode",
    "match_strategy",
    "selected_erm_session",
    "time_diff_hours",
    "data_input",
    "cov_path",
]
existing_columns = [column for column in columns if column in ready_noise_cov_jobs.columns]
ready_noise_cov_jobs[existing_columns]

## Write noise covariance matrices

Existing outputs are skipped unless `OVERWRITE_STEPS` contains `"noise_covariance"`. Missing inputs and per-recording failures are returned as status rows rather than aborting the whole batch.

In [ ]:
noise_cov_results = write_noise_covariances_for_recordings(
    config,
    selected_recordings,
    on_existing=noise_cov_policy,
    mode=NOISE_COV_MODE,
    method=COV_METHOD,
    rank=RANK,
    verbose=True,
)

noise_cov_results_df = noise_covariance_results_to_dataframe(noise_cov_results)
noise_cov_results_df

## Result summary

In [ ]:
if "noise_cov_results_df" not in globals() or noise_cov_results_df.empty:
    pd.DataFrame()
else:
    (
        noise_cov_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Inspect one covariance matrix

> Disabled by default for batch runs. Set `RUN_SINGLE_FILE_INSPECTIONS = True` in the selection cell to run this section.



In [ ]:
if RUN_COVARIANCE_QC_PLOTS:
    import mne
    import numpy as np
    import pandas as pd
    from pathlib import Path
    from IPython.display import display
    import matplotlib.pyplot as plt

    candidate_table_names = [
        "successful_results",
        "noise_cov_results_df",
        "candidate_results",
        "ready_noise_cov_jobs",
        "noise_cov_overview",
    ]

    cov_table = None
    cov_table_name = None

    for name in candidate_table_names:
        value = globals().get(name)
        if isinstance(value, pd.DataFrame) and not value.empty:
            if set(value.columns) & {"path", "cov_path", "output_path"}:
                cov_table = value.copy()
                cov_table_name = name
                break

    if cov_table is None:
        raise RuntimeError(
            "Could not find a covariance table. "
            "Run the covariance overview/computation cells first."
        )

    path_column = next(
        column
        for column in ["path", "cov_path", "output_path"]
        if column in cov_table.columns
    )

    def resolve_cov_path(value):
        path = Path(value)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        return path

    cov_table["_resolved_cov_path"] = cov_table[path_column].map(resolve_cov_path)
    cov_table["_cov_file_exists"] = cov_table["_resolved_cov_path"].map(
        lambda path: path.exists()
    )

    print(f"Using covariance table: {cov_table_name}")
    display(cov_table)

    cov_table = cov_table[cov_table["_cov_file_exists"]].copy()

    if cov_table.empty:
        raise RuntimeError(
            "No existing covariance files to inspect. "
            "Run the covariance writing cell first, or check the covariance paths above."
        )

    if MAX_COVARIANCE_QC_PLOTS is not None:
        cov_table = cov_table.head(MAX_COVARIANCE_QC_PLOTS).copy()

    print(f"Plotting {len(cov_table)} covariance file(s).")
    display(cov_table)

    def find_raw_bids_info_path(row: pd.Series) -> Path:
        subject = str(row["subject"]).removeprefix("sub-")
        task = str(row["task"])

        session_value = row.get("session", None)
        session = None
        if pd.notna(session_value) and str(session_value) not in {"", "None", "nan"}:
            session = str(session_value).removeprefix("ses-")

        run_value = row.get("run", None)
        run = None
        if pd.notna(run_value) and str(run_value) not in {"", "None", "nan"}:
            run = str(run_value).removeprefix("run-")

        if session is None:
            raw_dir = PROJECT_ROOT / f"sub-{subject}" / "meg"
            raw_pattern = f"sub-{subject}_task-{task}"
        else:
            raw_dir = PROJECT_ROOT / f"sub-{subject}" / f"ses-{session}" / "meg"
            raw_pattern = f"sub-{subject}_ses-{session}_task-{task}"

        if run is not None:
            raw_pattern += f"_run-{run}"

        raw_pattern += "_meg.fif"

        raw_candidates = sorted(raw_dir.glob(raw_pattern))

        if not raw_candidates:
            raw_candidates = sorted(
                (PROJECT_ROOT / f"sub-{subject}").glob(f"**/*task-{task}*_meg.fif")
            )

        if not raw_candidates:
            raise FileNotFoundError(
                f"Could not find raw BIDS FIF for subject={subject}, task={task}."
            )

        return raw_candidates[0]

    qc_rows = []

    for _, row in cov_table.iterrows():
        label_parts = [
            str(row.get("subject", "")),
            str(row.get("session", "")),
            str(row.get("task", "")),
            str(row.get("run", "")),
        ]
        label = " ".join(
            part for part in label_parts
            if part and part not in {"nan", "None"}
        )

        cov_path = row["_resolved_cov_path"]

        print("=" * 80)
        print(label)
        print("cov_path:", cov_path)

        try:
            raw_info_path = find_raw_bids_info_path(row)

            print("raw_info_path:", raw_info_path)

            cov = mne.read_cov(cov_path, verbose=False)
            info = mne.io.read_info(raw_info_path, verbose=False)
            cov_data = cov["data"]

            matrix_rank = np.linalg.matrix_rank(cov_data)
            max_abs_cov = np.abs(cov_data).max()
            median_abs_cov = np.median(np.abs(cov_data))

            print(cov)
            print("Number of channels:", len(cov["names"]))
            print("Covariance dimension:", cov["dim"])
            print("Matrix rank:", matrix_rank)
            print("n_samples:", cov.get("nfree", "unknown"))
            print("max_abs_cov:", max_abs_cov)
            print("median_abs_cov:", median_abs_cov)

            cov.plot(info, show_svd=True)
            plt.show()

            qc_rows.append(
                {
                    "subject": row.get("subject"),
                    "session": row.get("session"),
                    "task": row.get("task"),
                    "run": row.get("run"),
                    "status": "plotted",
                    "cov_path": cov_path,
                    "raw_info_path": raw_info_path,
                    "n_channels": len(cov["names"]),
                    "dim": cov["dim"],
                    "matrix_rank": matrix_rank,
                    "n_samples": cov.get("nfree", None),
                    "max_abs_cov": max_abs_cov,
                    "median_abs_cov": median_abs_cov,
                    "message": "",
                }
            )

        except Exception as error:
            print(f"FAILED: {error}")
            qc_rows.append(
                {
                    "subject": row.get("subject"),
                    "session": row.get("session"),
                    "task": row.get("task"),
                    "run": row.get("run"),
                    "status": "failed",
                    "cov_path": cov_path,
                    "raw_info_path": None,
                    "n_channels": None,
                    "dim": None,
                    "matrix_rank": None,
                    "n_samples": None,
                    "max_abs_cov": None,
                    "median_abs_cov": None,
                    "message": str(error),
                }
            )

    covariance_qc_plot_status = pd.DataFrame(qc_rows)
    display(covariance_qc_plot_status)
else:
    print("Skipped covariance QC plots.")